# Simplified Chinese support and reach on Steam

**DATA2010 Data Science Programming · final project**

The notebook follows the same sections as the README.

| Section | What it does | Output |
| :--- | :--- | :--- |
| I. Setup and how to reproduce | how to get the project and run it | |
| II.1 Business understanding | the question | |
| II.2 Data collection | read the two raw files | |
| II.3 Data processing | 3.1 clean · 3.2 SQ1 groups · 3.3 SQ2 groups · 3.4 save and check the tables | `pcd_games_clean`, `pcd_sq1_groups`, `pcd_sq2_groups` |
| II.4 Data analysis / EDA | describe the three tables | `report/figures/` |
| II.5 Data modelling | compare the groups, answer SQ1-SQ3 | `report/figures/` |
| II.6 Model deployment | | |

**How to read it**
- Each code cell = one decision, labelled 2.1, 2.2, ..., 3.1a, 3.1b, ...
- Its first lines say: `Reason` (why), `Rule` (what is done), `Not used ... because` (what was left out).
- `DATA ISSUE` = a problem found in the data. **Check** = the numbers a correct run shows.

# I. Setup and how to reproduce

**Needed:** [Git](https://git-scm.com) with [Git LFS](https://git-lfs.com) (the Windows installer of Git already includes it), Python 3.12 or 3.13.

**Steps**
1. `git lfs install` (once per computer)
2. `git clone https://github.com/theonlypgk/data2010-steam-chinese.git`
3. `cd data2010-steam-chinese` then `pip install -r requirements.txt`
4. Open `scripts/steam_chinese_analysis.ipynb` and choose *Run All*.

- `data/raw/games_a.csv` is only a few lines of text (not ~400 MB)? → run `git lfs pull` in the project folder.

In [ ]:
# I.a · Import the libraries and name the folders
# Reason: three libraries are enough; paths are relative to this notebook (scripts/)

import re                     # text patterns (clean the list cells, find Chinese characters)

import numpy as np
import pandas as pd

RAW = "../data/raw/"                          # the two input files
PROCESSED_CSV = "../data/processed/csv/"      # output tables
FIGURES = "../report/figures/"                # saved figures
PREFIX = "pcd_"                               # file prefix of every output table
print("pandas", pd.__version__, "| numpy", np.__version__)

# II. Data science life cycle

## 1.​ Business understanding

**Main question:** should a Steam game support Simplified Chinese, for which genres, and when?

| | Question |
|---|---|
| SQ1 | How much larger is the reach of games with Simplified Chinese, and in which genres? |
| SQ2 | Does adding Simplified Chinese after launch still increase reach, and in which genres? |
| SQ3 | For each genre: translate at launch, later, or not a priority? |

Context and the definition of reach: see the [README, II.1](https://github.com/theonlypgk/data2010-steam-chinese#1-business-understanding).

## 2.​ Data collection

- **Data:** Steam Games Dataset by FronkonGames (Hugging Face `FronkonGames/steam-games-dataset`), one row per game.
- **Two snapshots:** A = 2026-06-20, C = 2024-01-11, saved in `data/raw/` → same input on every run.
- **DATA ISSUE:** header of A merges two names (`DiscountDLC count`) → later columns shifted → repaired before reading.

### 2.1 The two snapshots

In [ ]:
# 2.1 · The two snapshots
# Reason: the dataset keeps changing -> the two files used are saved in data/raw
# Source: Hugging Face FronkonGames/steam-games-dataset, file games.csv
#   A = commit 52ba62abfa707a2fa31f4d147c2777a9d9fbd2da; C = commit aefdeb1ff1a739b92ac5338682cc2610a0328505

FILE_A = RAW + "games_a.csv"
FILE_C = RAW + "games_c.csv"
SNAPSHOT_DATE_A = "2026-06-20"                # day snapshot A was taken
SNAPSHOT_DATE_C = "2024-01-11"                # day snapshot C was taken

### 2.2 Read the two snapshots

In [ ]:
# 2.2 · Read the two snapshots
# DATA ISSUE: header of A merges two names ("DiscountDLC count") -> split it back, else columns shift
# Rule: read everything as text (dtype=str); fix types in 3.1a

raw_c = pd.read_csv(FILE_C, dtype=str)

names_a = pd.read_csv(FILE_A, nrows=0).columns.tolist()     # header only
i = names_a.index("DiscountDLC count")
names_a[i:i + 1] = ["Discount", "DLC count"]                 # 1 name -> 2 names
raw_a = pd.read_csv(FILE_A, names=names_a, skiprows=1, dtype=str)

### 2.3 Keep the 13 columns the analysis uses

In [ ]:
# 2.3 · Keep the 13 columns the analysis uses, with English names
# Reason: 27 of the 40 columns do not serve the questions

# Columns kept -> used for:
#   AppID                 -> id; join C to A
#   Name                  -> check results by eye
#   Release date          -> year band; ">= 1 year old" filter
#   Price, Discount       -> price tier (list price)
#   Supported languages   -> treatment; other languages; Chinese-first rule 1; languages added (3.3)
#   Full audio languages  -> voice-over signal, only to test the filter (3.1n, 5.1b)
#   Positive, Negative    -> outcome (reach)
#   Developers            -> Chinese-first rule 2
#   Publishers            -> shown in the 3.1m check
#   Genres                -> Early Access flag
#   Tags                  -> genre group
# Not kept because:
#   Estimated owners                         -> ranges too wide
#   scores, playtime, CCU, achievements      -> measure quality/engagement, not reach
#   Categories, age, DLC, platforms, notes   -> not part of the design
#   descriptions, links, media               -> no numbers
RAW_TO_CLEAN = {
    "AppID": "app_id",
    "Name": "name",
    "Release date": "release_date",
    "Price": "price",
    "Supported languages": "supported_languages",
    "Full audio languages": "full_audio_languages",
    "Positive": "positive",
    "Negative": "negative",
    "Developers": "developers",
    "Publishers": "publishers",
    "Genres": "genres",
    "Tags": "tags",
}

raw_a = raw_a[list(RAW_TO_CLEAN) + ["Discount"]].rename(columns=RAW_TO_CLEAN | {"Discount": "discount_pct"})
raw_c = raw_c[list(RAW_TO_CLEAN)].rename(columns=RAW_TO_CLEAN)
raw_c["discount_pct"] = "0"                    # C has no Discount column
print(f"snapshot A: {len(raw_a):,} games x {raw_a.shape[1]} columns")
print(f"snapshot C: {len(raw_c):,} games x {raw_c.shape[1]} columns")
raw_a.head(3)

**Check.** A has 125,855 games, C has 85,103; 13 columns each.

## 3.​ Data processing

| Part | What | Output |
|---|---|---|
| 3.1 Clean data | fix data issues, add analysis columns, filter | `pcd_games_clean` |
| 3.2 SQ1 groups | similar games, with vs without Chinese | `pcd_sq1_groups` |
| 3.3 SQ2 groups | similar games, added Chinese vs did not | `pcd_sq2_groups` |
| 3.4 Save and check the tables | write the output files, run automatic tests | |

- **Rule:** snapshots A and C are cleaned the same way.

### 3.1 Clean data

Four blocks, each = its decisions, then one code cell:

| Block | Steps | Output |
|---|---|---|
| Fix and add columns | 3.1a-h | `feat_a`, `feat_c` (all games, new columns) |
| Filter the rows | 3.1i-j | the filters |
| Build the table | 3.1k-l | `games_clean` |
| Check the cleaning | 3.1m-n | shown only |

#### Fix and add columns (3.1a-h)

**3.1a · Fix types and wrong single values**
- **DATA ISSUE:** sale prices in A (Hades 7.49 = 24.99 at -70%) → list price.
- **DATA ISSUE:** no id or repeated id → row dropped. Unreadable date → missing.

**3.1b · Clean the list cells**
- **DATA ISSUE:** junk in list cells (`English[b]*[/b]`, HTML) → removed.

**3.1c · Outcome, treatment, other languages**
- Outcome: `total_reviews = positive + negative`.
- Treatment: supports Simplified Chinese.
- Confounder: `n_other_langs`.

**3.1d · Flag the Chinese-first games**
Flagged here, removed in 3.1j.
- **Reason:** original language is Chinese → not a translation → outside the question.
- **Rules:** a game is flagged if
  1. it has Chinese text without English text, or
  2. every developer's name is in Chinese characters, without kana.
- **Not used:**
  - hand-made lists of cities or companies because: dataset too large → cannot check row by row.
  - voice-over languages because: Japanese games are dubbed in Chinese too → right only 1 of 9 times (3.1n).
  - names with kana because: kana = Japanese.
  - publishers because: often regional partners.
- **Missed:** Chinese studios with an English name and an English release (Black Myth: Wukong) → tested in 5.1b.

**3.1e · Flag Early Access and Traditional-Chinese-only games**
Flagged here, removed in 3.1j.
- Early Access because: unfinished game.
- Traditional Chinese only because: neither "with" nor "without" Simplified Chinese.

**3.1f · One genre group per game**
- Confounder: `genre_group`, because genre changes audience size and amount of text.
- **Rule:** first user tag found in the map (Steam orders tags by votes).

**3.1g · Bin and encode price, year, other languages**
- Confounders: `price_tier`, `year_band`, `n_other_langs_bin`.

**Encoding**
- **Rule:** bins saved as ranks 1, 2, 3, ...
- **Reason:** text labels sort wrongly ("10to20" before "5to10").
- Genre is not encoded because: genres have no order.

| Column | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| `price_tier` | free | < \$5 | \$5-10 | \$10-20 | > \$20 |
| `year_band` | <=2017 | 2018-2019 | 2020-2021 | 2022+ | |
| `n_other_langs_bin` | 0 | 1-4 | 5+ | | |

**3.1h · Run**
- The same steps on both snapshots → a game is measured the same way in A and C.

In [ ]:
# ----------------------------------------------------------------------------
# 3.1a · Fix the types and the wrong single values (DATA ISSUES)
# Reason: ids, counts, prices, dates arrive as text; some are wrong
# Rules: repeated id -> keep first; no id -> drop; sale price -> list price; unreadable date -> missing

# List price = sale price / (1 - discount / 100). Discount capped at 99% because: 100% -> divide by zero.
def fix_types(trim, snapshot_date):
    """Give one snapshot real types, one row per game, list prices and dates.

    Args:
        trim: one snapshot from 2.3 (13 columns, every value is text).
        snapshot_date: day the snapshot was taken, "YYYY-MM-DD".

    Returns:
        A new DataFrame, one row per app_id:
        app_id, positive, negative -> int64; price -> list price (float);
        release_date -> datetime (NaT if unreadable); new columns
        price_at_snapshot, release_year, days_since_release.
    """
    df = trim.copy()
    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce")        # bad id -> NaN
    df = df.dropna(subset=["app_id"]).drop_duplicates("app_id")        # DATA ISSUE: no / repeated id
    df["app_id"] = df["app_id"].astype("int64")
    for col in ("positive", "negative"):
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int64")
    # DATA ISSUE: sale prices in A -> list price = price / (1 - discount / 100)
    df["price_at_snapshot"] = pd.to_numeric(df["price"], errors="coerce")
    discount = df["discount_pct"] if "discount_pct" in df else pd.Series(0, index=df.index)
    df["discount_pct"] = pd.to_numeric(discount, errors="coerce").fillna(0).clip(0, 99)
    df["price"] = (df["price_at_snapshot"] / (1 - df["discount_pct"] / 100)).round(2)
    # DATA ISSUE: dates come as "Aug 1, 2023" or "Aug 2023"; unreadable -> NaT (missing)
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce", format="mixed")
    df["release_year"] = df["release_date"].dt.year.astype("Int64")
    df["days_since_release"] = (pd.Timestamp(snapshot_date) - df["release_date"]).dt.days
    return df


# ----------------------------------------------------------------------------
# 3.1b · Turn the list-like text cells into clean lists (DATA ISSUE)
# Reason: languages, tags, genres are text that looks like a list, with store-page junk

# DATA ISSUE: store-page leftovers in list cells: BBCode tags, HTML tags/entities, "*", line breaks
_TAG_JUNK = re.compile(r"\[/?[a-zA-Z]+\]|<[^>]+>|&amp;|&lt;|&gt;|\*|\r|\n|#lang_\w+")


# Handles: "['A', 'B']", "A,B", junk ("English[b]*[/b]"), missing. Keeps the order because: tags are sorted by votes.
def parse_list(value):
    """Turn one list-like text cell into a clean list.

    Args:
        value: a cell such as "['English', 'French']", "A,B", "English[b]*[/b]" or NaN.

    Returns:
        List of unique strings in the original order, e.g. ["English", "French"];
        [] for a missing cell.
    """
    if pd.isna(value):
        return []                                          # missing cell
    text = _TAG_JUNK.sub("", str(value)).strip()           # remove the junk first
    items = text.strip("[]").split(",")                    # "['A', 'B']" or "A,B" -> pieces
    out = []
    for item in items:
        item = item.strip().strip("'\"").strip()           # drop spaces and quotes
        if item and item != "nan" and item not in out:     # drop empties and repeats, keep the order
            out.append(item)
    return out


# ----------------------------------------------------------------------------
# 3.1c · Outcome, treatment and the other-languages confounder
# Reason: Steam publishes no sales -> reach = number of reviews
# Rule: treatment = supports Simplified Chinese; also count the other languages (they affect reach)

# Languages: the treatment, and the language every Western game ships with.
ZH_SIMPLIFIED = "Simplified Chinese"
ZH_TRADITIONAL = "Traditional Chinese"
ENGLISH = "English"


def add_outcome_and_languages(df, langs):
    """Add the outcome and the language columns.

    Args:
        df: snapshot after fix_types.
        langs: Series of clean language lists (parse_list of supported_languages).

    Returns:
        The same DataFrame with new columns:
        total_reviews (int), has_zh_simplified, has_zh_traditional, has_english (bool),
        n_other_langs (int), all_langs (str, e.g. "English|French").
    """
    df["total_reviews"] = df["positive"] + df["negative"]
    chinese = {ZH_SIMPLIFIED, ZH_TRADITIONAL}
    df["has_zh_simplified"] = langs.map(lambda x: ZH_SIMPLIFIED in x)
    df["has_zh_traditional"] = langs.map(lambda x: ZH_TRADITIONAL in x)
    df["has_english"] = langs.map(lambda x: ENGLISH in x)
    df["n_other_langs"] = langs.map(lambda x: len(set(x) - chinese - {ENGLISH}))   # set difference
    df["all_langs"] = langs.map(lambda x: "|".join(sorted(x)))                   # "English|French|..."
    return df


# ----------------------------------------------------------------------------
# 3.1d · Flag the Chinese-first games
# Reason: original language is Chinese -> not a translation -> outside the question
# Rules: (1) Chinese text without English text; (2) every developer name in Chinese characters, no kana
# Not used: hand-made lists of cities/companies because: dataset too large -> cannot check row by row
# Not used: voice-over languages because: Japanese games are dubbed in Chinese too -> right only 1 of 9 times (3.1n)
# Not used: names with kana because: kana = Japanese

# Legal suffixes that become separate items when a developer cell is split on commas.
COMPANY_SUFFIXES = ["inc", "ltd", "llc", "co", "corp", "limited", "gmbh", "plc", "sa", "srl", "co. ltd"]


_HAN = re.compile(r"[一-鿿]")       # a Chinese character (also used in Japanese kanji)
_KANA = re.compile(r"[぀-ヿ]")      # hiragana or katakana: only Japanese uses them


def split_developers(text):
    """Split a developer cell into developer names.

    Args:
        text: developer cell, e.g. "A, B, Inc.".

    Returns:
        List of names without legal suffixes, e.g. ["A", "B"].
    """
    parts = [p.strip().strip(".").strip() for p in str(text).split(",")]   # split on commas, trim spaces and dots
    suffixes = set(COMPANY_SUFFIXES)
    return [p for p in parts if p and p.lower().rstrip(".") not in suffixes]   # drop empty parts and suffixes


# "All" developers, not "any" because: a foreign game can list one Chinese co-developer.
def all_match(developers, pattern):
    """Check that every developer name matches a pattern.

    Args:
        developers: list of names from split_developers.
        pattern: compiled regex, e.g. _HAN (Chinese characters).

    Returns:
        True if the list is not empty and every name matches; otherwise False.
    """
    return bool(developers) and all(pattern.search(d) for d in developers)   # False for an empty list


# cn_text      Chinese text without English text                    (signal 1, used)
# cn_dev_name  all developer names in Chinese characters, no kana   (signal 2, used)
# cn_dev_kana  the same, with kana = Japanese                       (not used; tested in 3.1n, 5.1b)
# cn_audio     Chinese voice-over without English voice-over        (not used; tested in 3.1n, 5.1b)
def add_chinese_game_flags(df, audio):
    """Flag the Chinese-first games.

    Args:
        df: snapshot after add_outcome_and_languages.
        audio: Series of clean voice-over language lists.

    Returns:
        The same DataFrame with new columns:
        cn_text, cn_dev_name, cn_dev_kana, cn_audio (bool, the 4 signals),
        is_cn_game (bool, cn_text or cn_dev_name), cn_rules (str, e.g. "cn_text;cn_dev_name"),
        makers (str, "developers | publishers").
    """
    chinese = {ZH_SIMPLIFIED, ZH_TRADITIONAL}
    developers = df["developers"].fillna("").astype(str)
    publishers = df["publishers"].fillna("").astype(str)
    dev_lists = developers.map(split_developers)
    # (Python's re via .map, not .str.contains: with pyarrow installed pandas uses Arrow's
    # regex engine, which rejects some of these patterns.)
    han_name = dev_lists.map(lambda d: all_match(d, _HAN))             # all developers: Chinese characters
    kana_name = developers.map(lambda text: bool(_KANA.search(text)))  # any kana in the developer cell
    has_zh_audio = audio.map(lambda x: bool(set(x) & chinese))
    has_en_audio = audio.map(lambda x: ENGLISH in x)

    df["cn_text"] = (df["has_zh_simplified"] | df["has_zh_traditional"]) & ~df["has_english"]
    df["cn_dev_name"] = han_name & ~kana_name
    df["cn_dev_kana"] = han_name & kana_name
    df["cn_audio"] = has_zh_audio & ~has_en_audio
    df["is_cn_game"] = df["cn_text"] | df["cn_dev_name"]               # the filter of 3.1j

    used = df[["cn_text", "cn_dev_name"]]
    # names of the matched signals joined by ";", for the check in 3.1m
    df["cn_rules"] = used.apply(lambda r: ";".join(r.index[r.to_numpy()]), axis=1)
    df["makers"] = developers + " | " + publishers
    return df


# ----------------------------------------------------------------------------
# 3.1e · Flag Early Access and Traditional-Chinese-only games
# Reason: Early Access = unfinished game; Traditional-only = neither "with" nor "without" Simplified Chinese
# Rule: flag here, remove in 3.1j

def add_filter_flags(df, genres):
    """Flag the games removed later in 3.1j.

    Args:
        df: snapshot after add_outcome_and_languages.
        genres: Series of clean genre lists.

    Returns:
        The same DataFrame with new columns is_early_access and is_traditional_only (bool).
    """
    df["is_early_access"] = genres.map(lambda x: "Early Access" in x)
    df["is_traditional_only"] = df["has_zh_traditional"] & ~df["has_zh_simplified"]
    return df


# ----------------------------------------------------------------------------
# 3.1f · Give each game one genre group
# Reason: genre changes audience size and amount of text
# Rule: first user tag found in the map (Steam orders tags by votes)

# Genre groups: user tag -> group; the first matching tag of a game wins.
TAG_TO_GENRE = {
    "Visual Novel": "Visual Novel",
    "RPG": "RPG", "JRPG": "RPG", "CRPG": "RPG", "Action RPG": "RPG", "Party-Based RPG": "RPG",
    "Strategy": "Strategy", "Turn-Based Strategy": "Strategy", "RTS": "Strategy",
    "Grand Strategy": "Strategy", "4X": "Strategy", "Tower Defense": "Strategy",
    "Action": "Action", "Shooter": "Action", "FPS": "Action", "Platformer": "Action",
    "Hack and Slash": "Action", "Fighting": "Action",
    "Puzzle": "Puzzle", "Hidden Object": "Puzzle",
    "Casual": "Casual",
}


def first_genre(tags):
    """Give a game one genre group.

    Args:
        tags: clean list of user tags, most voted first.

    Returns:
        Group of the first tag found in TAG_TO_GENRE, e.g. "RPG"; None if no tag matches.
    """
    for tag in tags:
        if tag in TAG_TO_GENRE:
            return TAG_TO_GENRE[tag]
    return None


# ----------------------------------------------------------------------------
# 3.1g · Bin price, release year and other languages, and encode the bins
# Reason: groups need a few levels, not exact values
# Rule: save bins as ranks 1, 2, 3, ... because: text labels sort wrongly ("10to20" before "5to10")

# Bins; each label is saved as its position 1, 2, 3, ... in the list.
PRICE_BINS = [-0.001, 0.0, 4.999, 9.999, 19.999, float("inf")]    # USD, list price
PRICE_LABELS = ["free", "lt5", "5to10", "10to20", "gt20"]           # codes 1-5
YEAR_BAND_BINS = [0, 2017, 2019, 2021, 2100]
YEAR_BAND_LABELS = ["<=2017", "2018-2019", "2020-2021", "2022+"]    # codes 1-4
OTHER_LANG_BINS = [-1, 0, 4, 10_000]
OTHER_LANG_LABELS = ["0", "1-4", "5+"]                              # codes 1-3


def to_code(values, bins):
    """Put numbers into bins and return the bin number.

    Args:
        values: Series of numbers, e.g. prices.
        bins: bin edges, e.g. PRICE_BINS.

    Returns:
        Series of Int64 codes 1, 2, 3, ... (missing stays missing).
    """
    # labels=False gives the bin position 0, 1, 2, ...; +1 makes it 1, 2, 3, ...
    return (pd.cut(values.astype("float"), bins, labels=False) + 1).astype("Int64")


def add_codes(df):
    """Add the binned confounders as codes.

    Args:
        df: snapshot with price, release_year and n_other_langs.

    Returns:
        The same DataFrame with new columns price_tier (1-5), year_band (1-4),
        n_other_langs_bin (1-3), all Int64.
    """
    df["price_tier"] = to_code(df["price"], PRICE_BINS)                # codes 1-5
    df["year_band"] = to_code(df["release_year"], YEAR_BAND_BINS)      # codes 1-4
    df["n_other_langs_bin"] = to_code(df["n_other_langs"], OTHER_LANG_BINS)   # codes 1-3
    return df


# ----------------------------------------------------------------------------
# 3.1h · Run: clean both snapshots the same way
# Reason: one function for both snapshots -> a game is measured the same way in A and C

def add_features(trim, snapshot_date):
    """Run steps 3.1a-g on one snapshot.

    Args:
        trim: one snapshot from 2.3.
        snapshot_date: day the snapshot was taken, "YYYY-MM-DD".

    Returns:
        DataFrame, one row per game (no game removed yet), with every column
        added in 3.1a-g.
    """
    df = fix_types(trim, snapshot_date)                               # 3.1a
    langs = df["supported_languages"].map(parse_list)                 # 3.1b
    audio = df["full_audio_languages"].map(parse_list)
    tags = df["tags"].map(parse_list)
    genres = df["genres"].map(parse_list)
    df = add_outcome_and_languages(df, langs)                         # 3.1c
    df = add_chinese_game_flags(df, audio)                            # 3.1d
    df = add_filter_flags(df, genres)                                 # 3.1e
    df["genre_group"] = tags.map(first_genre)                         # 3.1f
    return add_codes(df)                                              # 3.1g


feat_a = add_features(raw_a, SNAPSHOT_DATE_A)
feat_c = add_features(raw_c, SNAPSHOT_DATE_C)
print(f"A: {len(feat_a):,} games, C: {len(feat_c):,} games")

**Check.** No game removed yet: A = 125,855, C = 85,103.

#### Filter the rows (3.1i-j)

| Rule | Reason |
|---|---|
| drop Chinese-first games | not a translation |
| drop Early Access | unfinished game |
| keep >= 10 reviews | below that, reach is noise |
| keep released >= 1 year before the snapshot | time to collect reviews |
| keep games with genre, price and year | needed to form groups |
| `pcd_games_clean` only: drop Traditional-Chinese-only | neither "with" nor "without" Simplified Chinese |

- Not a higher review threshold because: it would filter on the outcome itself.

In [ ]:
# ----------------------------------------------------------------------------
# 3.1i · Apply filters in order and count what each removes
# Reason: the funnel must show what each filter removed

def apply_filters(df, steps, table):
    """Apply filters in order and count the games left after each one.

    Args:
        df: table to filter.
        steps: list of (filter name, Series True = keep), applied in order.
        table: name written in the funnel, e.g. "games_clean".

    Returns:
        (kept, funnel): kept = copy of the rows that pass every filter;
        funnel = DataFrame with columns table, step, games_left (one row per step).
    """
    rows = [{"table": table, "step": "start", "games_left": len(df)}]
    keep = pd.Series(True, index=df.index)
    for name, mask in steps:
        keep &= mask.reindex(df.index).fillna(False)       # missing value = fails the filter
        rows.append({"table": table, "step": name, "games_left": int(keep.sum())})
    return df[keep].copy(), pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# 3.1j · The filters shared by both snapshots, and their thresholds
# Rules: >= 10 reviews because: below that, reach is noise
#        >= 1 year old because: time to collect reviews

MIN_REVIEWS = 10              # below that, reach is noise (see the table above)
MIN_AGE_DAYS = 365            # every game had a year to collect reviews


def base_filters(df):
    """List the filters shared by both snapshots.

    Args:
        df: snapshot after add_features.

    Returns:
        List of (filter name, Series True = keep), in the order they are applied.
    """
    return [
        ("drop Chinese-first games (either of 2 signals)", ~df["is_cn_game"]),
        ("drop Early Access", ~df["is_early_access"]),
        (f"keep >= {MIN_REVIEWS} reviews", df["total_reviews"] >= MIN_REVIEWS),
        ("keep released >= 1 year before snapshot", df["days_since_release"] >= MIN_AGE_DAYS),
        ("keep games with a genre group, price and year",
         df["genre_group"].notna() & df["price"].notna() & df["release_year"].notna()),
    ]

#### Build `pcd_games_clean` (3.1k-l)

- One row per game of snapshot A that passed every filter.
- `has_zh`: 1 = Simplified Chinese, 0 = no Chinese.
- `log_total_reviews` added because: review counts are very skewed.

In [ ]:
# ----------------------------------------------------------------------------
# 3.1k · Build the cleaned dataset pcd_games_clean
# Reason: one row per game, only the columns the analysis uses

def build_games_clean(feat_a):
    """Build pcd_games_clean from snapshot A.

    Args:
        feat_a: snapshot A after add_features.

    Returns:
        (games_clean, funnel): games_clean = one row per game that passed every filter,
        12 columns (app_id, name, genre_group, release_year, year_band, price, price_tier,
        n_other_langs, n_other_langs_bin, has_zh 0/1, total_reviews, log_total_reviews);
        funnel = games left after each filter.
    """
    steps = base_filters(feat_a) + [("drop Traditional-Chinese-only", ~feat_a["is_traditional_only"])]
    df, funnel = apply_filters(feat_a, steps, "games_clean")
    df["has_zh"] = df["has_zh_simplified"].astype(int)               # True/False -> 1/0
    df["log_total_reviews"] = np.log10(df["total_reviews"]).round(6) # rounded: same bytes every run
    for col in ["price_tier", "year_band", "n_other_langs_bin"]:
        df[col] = df[col].astype(int)                                # no missing left after filters
    cols = [
        "app_id", "name", "genre_group", "release_year", "year_band", "price", "price_tier",
        "n_other_langs", "n_other_langs_bin", "has_zh", "total_reviews", "log_total_reviews",
    ]
    return df[cols].sort_values("app_id").reset_index(drop=True), funnel


# ----------------------------------------------------------------------------
# 3.1l · Run: build the cleaned dataset
# Reason: snapshot A = current catalogue, used for SQ1

games_clean, funnel_a = build_games_clean(feat_a)
print(f"games_clean: {len(games_clean):,} games, {games_clean['has_zh'].sum():,} with Simplified Chinese")
games_clean.head()

**Check.** `games_clean` has 45,445 games, 14,701 of them with Simplified Chinese.

#### Check the cleaning (3.1m-n)

**3.1m · What the cleaning removed**
- Funnel: games left after each filter.
- List: the biggest Chinese-first games removed.
- Shown only, not saved.

**3.1n · Hand check of a random sample**
- **Reason:** dataset too large → cannot check row by row.
- **Rule:** 50 random games with any of the four signals, each checked by hand (developer, store page).

| Signal | Used? | Games | Chinese-first | Not | Unclear |
|---|---|---:|---:|---:|---:|
| `cn_text`: Chinese text, no English text | yes | 25 | 25 | 0 | 0 |
| `cn_dev_name`: names in Chinese characters, no kana | yes | 12 | 11 | 1 | 0 |
| `cn_dev_kana`: names with kana | no | 4 | 0 | 4 | 0 |
| `cn_audio`: Chinese voice-over, no English voice-over | no | 9 | 1 | 4 | 4 |

In [ ]:
# ----------------------------------------------------------------------------
# 3.1m · Show the funnel and the biggest Chinese-first games removed
# Reason: spot a filter that removes too much, or a wrongly removed game

cn_cols = ["app_id", "name", "makers", "cn_rules", "has_english", "total_reviews"]
cn_excluded = (
    feat_a[feat_a["is_cn_game"] & (feat_a["total_reviews"] >= MIN_REVIEWS)][cn_cols]  # would pass the review filter
    .sort_values("total_reviews", ascending=False)                                   # biggest first
    .reset_index(drop=True)
)
print(f"{len(cn_excluded):,} Chinese-first games removed (>= {MIN_REVIEWS} reviews)")
display(funnel_a)
display(cn_excluded.head(20))


# ----------------------------------------------------------------------------
# 3.1n · Draw the random sample that was checked by hand
# Reason: dataset too large to check row by row -> check a random sample
# Rule: 50 games with any of the 4 signals; fixed seed -> same games every run

AUDIT_SIZE = 50               # games checked by hand
AUDIT_SEED = 2010             # fixed draw: the same sample on every run
SIGNALS = ["cn_text", "cn_dev_name", "cn_dev_kana", "cn_audio"]     # in the order of the table above
any_signal = feat_a[feat_a[SIGNALS].any(axis=1) & (feat_a["total_reviews"] >= MIN_REVIEWS)]
audit = any_signal.sample(n=AUDIT_SIZE, random_state=AUDIT_SEED).copy()
# idxmax on True/False columns gives the first column that is True: the first signal that applies
audit["signal"] = audit[SIGNALS].idxmax(axis=1)
print(f"{AUDIT_SIZE} of {len(any_signal):,} games with any signal, drawn with random_state={AUDIT_SEED}")
print(audit["signal"].value_counts().reindex(SIGNALS).to_string())
audit[["app_id", "name", "makers", "signal", "has_english", "total_reviews"]]

**Result (3.1m)**
- Biggest drop: the review filter (110,107 → 50,160) because: most Steam games have < 10 reviews.
- 2,030 Chinese-first games removed; top of the list: Tale of Immortal, The Scroll Of Taiwu, The Hungry Lamb.

**Result (3.1n)**
- Used signals: right for 36 of 37 games.
- Unused signals: right for 1 of 13 → not used.
- A wrongly removed game only leaves the sample → no bias in the comparison.

### 3.2 SQ1 groups → `pcd_sq1_groups`

- **Prepares for SQ1:** how much larger is the reach of games with Simplified Chinese, and in which genres?
- **Rule:** compare games with and without Chinese only inside groups of similar games.
- **Reason:** bigger games are both translated more and reviewed more → a direct comparison would mix the two.

#### 3.2a · Define the SQ1 comparison group

**Group = genre x price tier x year band x other languages**

| In the group | Reason |
|---|---|
| Genre | audience size and amount of text differ |
| Price tier | stands in for budget and studio size |
| Year band | older games had more time to collect reviews |
| Other languages | games with Chinese: median 7 other languages; without: 0 |

- Year bands, not single years because: single years → only 47% of games in usable groups (bands: 79%).

In [ ]:
# 3.2a · Define the SQ1 comparison group
# Reason: compare games only inside groups of similar games

# code -> label, to write readable group names ("Action | <=2017 | lt5 | 5+")
PRICE_LABEL = dict(enumerate(PRICE_LABELS, start=1))
YEAR_BAND_LABEL = dict(enumerate(YEAR_BAND_LABELS, start=1))
OTHER_LANGS_LABEL = dict(enumerate(OTHER_LANG_LABELS, start=1))


def add_sq1_group(games):
    """Copy of games with group_id = genre | year band | price tier | other languages."""
    # assign returns a copy with the new column; the input table is not changed
    return games.assign(group_id=(
        games["genre_group"] + " | " + games["year_band"].map(YEAR_BAND_LABEL) + " | "
        + games["price_tier"].map(PRICE_LABEL) + " | " + games["n_other_langs_bin"].map(OTHER_LANGS_LABEL)
    ))

#### 3.2b · Count games and median reviews per side

**Table `pcd_sq1_groups`** (one row per group)
- Per side: number of games and median reviews. Medians because: review counts are very skewed.
- `eligible` = >= 10 games per side because: fewer → one game moves the median.
- 360 groups, 190 eligible, 79% of games.
- The comparison of the two sides is in 5.1.

In [ ]:
# 3.2b · Count games and median reviews on each side of each group
# Rules: medians because: review counts are very skewed
#        eligible = >= 10 games per side because: fewer -> one game moves the median

MIN_GAMES_PER_SIDE = 10       # a group is eligible only with this many games per side (also 3.3)


def build_sq1_groups(games):
    """One row per group: games and median reviews with / without Chinese, eligible."""
    keys = ["group_id", "genre_group", "year_band", "price_tier", "n_other_langs_bin"]
    wide = (
        games.groupby(keys + ["has_zh"])["total_reviews"]
        .agg(n="size", median="median")                    # count and median per group and side
        .unstack("has_zh")                                 # sides become columns: (stat, 0/1)
    )
    out = pd.DataFrame({
        "n_zh": wide[("n", 1)],
        "n_no_zh": wide[("n", 0)],
        "median_reviews_zh": wide[("median", 1)],
        "median_reviews_no_zh": wide[("median", 0)],
    }).reset_index()
    out[["n_zh", "n_no_zh"]] = out[["n_zh", "n_no_zh"]].fillna(0).astype(int)  # empty side -> 0
    out["eligible"] = (out["n_zh"] >= MIN_GAMES_PER_SIDE) & (out["n_no_zh"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values(keys[1:]).reset_index(drop=True)   # genre, then codes: old -> new, cheap -> expensive

In [ ]:
# 3.2c · Run: build the SQ1 group table
# Reason: one row per group = the table SQ1 is analysed from (5.1)

sq1_groups = build_sq1_groups(add_sq1_group(games_clean))
print(f"sq1_groups: {len(sq1_groups)} groups, {sq1_groups['eligible'].sum()} eligible")
sq1_groups.head()

**Check.** 360 groups, 190 of them eligible.

### 3.3 SQ2 groups → `pcd_sq2_groups`

- **Prepares for SQ2:** does adding Simplified Chinese after launch still increase reach, and in which genres?
- **Rule:** follow games with no Chinese in C (2024-01) to A (2026-06); compare those that added Chinese (treated) with similar games that did not (control).
- **Reason for a control:** every game gains reviews over time anyway.

#### 3.3a · Treated, control or excluded

- `treated` = added Simplified Chinese. `control` = added no Chinese and no major language.
- `excluded` = added another major language because: own localisation push → not a clean control.

In [ ]:
# 3.3a · Treated, control or excluded: what a game added between C and A
# Rules: treated = added Simplified Chinese; control = added no Chinese and no major language
# Excluded: added another major language because: own localisation push -> not a clean control

MAJOR_LANGS = [               # a control game must not have added any of these
    "German", "French", "Spanish - Spain", "Japanese", "Russian",
    "Korean", "Portuguese - Brazil",
]


# Languages in A but not in C = added after January 2024.
def label_arm(a, c):
    """treated / control / excluded for games without Chinese in C."""
    new_langs = [
        set(la.split("|")) - set(lc.split("|")) - {""}
        for la, lc in zip(a["all_langs"], c["all_langs"])
    ]
    added_zh = a["has_zh_simplified"].to_numpy()           # had no Chinese in C, has it in A
    added_major = np.array([bool(n & set(MAJOR_LANGS)) for n in new_langs])
    # np.select: first condition that is True gives the label, otherwise the default
    return np.select([added_zh, ~added_zh & ~added_major], ["treated", "control"], default="excluded")

#### 3.3b · Build the SQ2 panel

**Panel** (one row per game; not saved)
- Games: snapshot C, filtered as in 3.1, no Chinese in C, still in A.
- Characteristics from C because: the state before adding Chinese.
- 0 reviews in A → dropped because: delisted, -100% growth is not real.
- `growth = (reviews in A - reviews in C) / reviews in C`.

In [ ]:
# 3.3b · Build the SQ2 panel: the same games followed from C to A
# Rules: characteristics from C because: the state before adding Chinese
#        drop 0 reviews in A because: delisted -> -100% growth is not real

def build_sq2_panel(feat_a, feat_c):
    """One row per game of C (no Chinese, filtered as in 3.1) also present in A; and its funnel."""
    steps = base_filters(feat_c) + [
        ("keep games with no Chinese (Simplified or Traditional) in C",
         ~(feat_c["has_zh_simplified"] | feat_c["has_zh_traditional"])),
        ("keep games also present in A", feat_c["app_id"].isin(feat_a["app_id"])),
    ]
    c, funnel = apply_filters(feat_c, steps, "sq2_panel")
    c = c.set_index("app_id")
    a = feat_a.set_index("app_id").loc[c.index]            # same games, same order, from A

    p = pd.DataFrame(index=c.index)
    for col in ["name", "genre_group", "release_year", "year_band", "price_tier"]:
        p[col] = c[col]                                    # characteristics BEFORE: from C
    p["price_c"] = c["price"]
    p["reviews_c"] = c["total_reviews"]
    p["reviews_a"] = a["total_reviews"]
    p = p[p["reviews_a"] > 0].copy()                       # drop delisted games (0 reviews in A)
    a, c = a.loc[p.index], c.loc[p.index]
    p["growth"] = ((p["reviews_a"] - p["reviews_c"]) / p["reviews_c"]).round(6)

    p["arm"] = label_arm(a, c)                             # 3.3a: treated / control / excluded
    for col in ["year_band", "price_tier"]:
        p[col] = p[col].astype(int)                        # no missing left after filters
    funnel = pd.concat([funnel, pd.DataFrame([
        {"table": "sq2_panel", "step": "keep games with reviews in A (not delisted)", "games_left": len(p)},
        {"table": "sq2_panel", "step": "arm = treated", "games_left": int((p["arm"] == "treated").sum())},
        {"table": "sq2_panel", "step": "arm = control", "games_left": int((p["arm"] == "control").sum())},
    ])], ignore_index=True)
    cols = ["name", "genre_group", "release_year", "year_band", "price_c", "price_tier",
            "reviews_c", "reviews_a", "growth", "arm"]
    return p[cols].reset_index().sort_values("app_id").reset_index(drop=True), funnel

In [ ]:
# 3.3c · Run: build the panel and show its funnel
# Reason: the funnel shows how many games each step keeps

sq2_panel, funnel_c = build_sq2_panel(feat_a, feat_c)
print(f"sq2_panel: {len(sq2_panel):,} games")
print(sq2_panel["arm"].value_counts().to_string())
display(funnel_c)

**Check.** The panel has 21,590 games: 510 treated, 20,839 control and 241 excluded.

#### 3.3d · Count games and median growth per arm

**Group = genre x year band**

| Variable | In the group? | Reason |
|---|---|---|
| Genre | yes | needed to answer per genre |
| Year band | yes | control games grow +12% (<=2017) vs +77% (2022+); treated games are ~2 years newer |
| Price tier | no | little effect once the year is fixed; with it only 3 genres keep usable groups |
| Other languages | no | same mix in both arms |

- Only two variables because: only 510 games added Chinese.

**Table `pcd_sq2_groups`**
- Per arm: number of games and median growth.
- `eligible` = >= 10 games per side.
- 24 groups, 16 eligible, 90% of treated games, all 6 genres.
- The comparison of the two arms is in 5.2.

In [ ]:
# 3.3d · Count games and median growth of each arm inside each SQ2 group
# Rule: group = genre x year band because: the strongest confounders of growth

def add_sq2_group(panel):
    """Copy of the panel with group_id_sq2 = genre | year band."""
    # assign returns a copy with the new column; map turns the year code back into its label
    return panel.assign(group_id_sq2=panel["genre_group"] + " | " + panel["year_band"].map(YEAR_BAND_LABEL))


def build_sq2_groups(panel):
    """One row per group: treated and control games, their median growth, eligible."""
    keys = ["group_id_sq2", "genre_group", "year_band"]
    used = panel[panel["arm"].isin(["treated", "control"])]          # "excluded" is not compared
    # count and median growth per group and arm; unstack makes treated / control columns
    wide = used.groupby(keys + ["arm"])["growth"].agg(n="size", median="median").unstack("arm")
    out = pd.DataFrame({
        "n_treated": wide[("n", "treated")],
        "n_control": wide[("n", "control")],
        "median_growth_treated": wide[("median", "treated")].round(6),
        "median_growth_control": wide[("median", "control")].round(6),
    }).reset_index()
    out[["n_treated", "n_control"]] = out[["n_treated", "n_control"]].fillna(0).astype(int)  # empty arm -> 0
    out["eligible"] = (out["n_treated"] >= MIN_GAMES_PER_SIDE) & (out["n_control"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values(keys[1:]).reset_index(drop=True)          # genre, then old -> new

In [ ]:
# 3.3e · Run: build the SQ2 group table
# Reason: one row per group = the table SQ2 is analysed from (5.2)

sq2_groups = build_sq2_groups(add_sq2_group(sq2_panel))
print(f"sq2_groups: {len(sq2_groups)} groups, {sq2_groups['eligible'].sum()} eligible")
sq2_groups

**Check.** 24 groups, 16 of them eligible.

### 3.4 Save the tables and check them

- **Save:** write the three output tables to `data/processed/csv/`. (`data/processed/xlsx/` holds a copy to open and look at.)
- **Check:** automatic tests on the tables (ids unique, known games kept or removed, numbers equal to the ones reported). One fails → the notebook stops with an error.

In [ ]:
# 3.4a · Write the three tables as csv
# Rule: csv only because: every later step reads the csv
#   (data/processed/xlsx holds a copy of the same tables, only to open and look at)

tables = {
    "games_clean": games_clean,   # 3.1: cleaned data, one row per game
    "sq1_groups": sq1_groups,     # 3.2: one row per SQ1 group
    "sq2_groups": sq2_groups,     # 3.3: one row per SQ2 group
}
for name, table in tables.items():
    table.to_csv(PROCESSED_CSV + PREFIX + name + ".csv", index=False)
    print(f"{PREFIX}{name}: {len(table):,} rows x {table.shape[1]} columns")

In [ ]:
# 3.4b · Check the structure of the tables and the saved files
# Reason: catch broken ids, codes out of range, non-English column names, a csv that was not written

# structure: unique ids, codes in range, English snake_case column names
assert games_clean["app_id"].is_unique and sq2_panel["app_id"].is_unique
assert set(games_clean["has_zh"]) <= {0, 1}
assert set(sq2_panel["arm"]) <= {"treated", "control", "excluded"}
assert games_clean["price_tier"].between(1, 5).all() and games_clean["year_band"].between(1, 4).all()
assert games_clean["n_other_langs_bin"].between(1, 3).all()
for name, table in tables.items():
    assert all(re.fullmatch(r"[a-z0-9_]+", c) for c in table.columns), name

# saved files: the csv holds the same table
for name, table in tables.items():
    back = pd.read_csv(PROCESSED_CSV + PREFIX + name + ".csv")
    assert back.shape == table.shape and list(back.columns) == list(table.columns), name

In [ ]:
# 3.4c · Check the data issues, the rules and the groups on known cases
# Reason: known games show whether the rules behave as intended

# known games: data issues and rules behave as intended
by_name = games_clean.set_index("name")
assert by_name.loc["Hades", "price"] > 20                             # list price, not the 7.49 sale price
assert by_name.loc["Hades", "price_tier"] == PRICE_LABELS.index("gt20") + 1
for name in ["Cyberpunk 2077", "Far Cry® 5", "Core Keeper", "Hollow Knight",   # Western games
             "Ether One", "STEINS;GATE"]:              # a name like a Chinese company; Japanese, dubbed in Chinese
    assert name in by_name.index, f"{name} should be kept"
for name in ["鬼谷八荒 Tale of Immortal", "The Scroll Of Taiwu", "Sword and Fairy 7"]:
    assert name in set(cn_excluded["name"]), f"{name} should be removed"  # signal 2, signal 1, both
assert sq2_panel["growth"].min() > -1                                 # no delisted game left

# group tables: every game in exactly one group, eligible groups really have >= 10 games per side
assert sq1_groups[["n_zh", "n_no_zh"]].to_numpy().sum() == len(games_clean)
assert sq2_groups[["n_treated", "n_control"]].to_numpy().sum() == (sq2_panel["arm"] != "excluded").sum()
assert (sq1_groups.loc[sq1_groups["eligible"], ["n_zh", "n_no_zh"]] >= MIN_GAMES_PER_SIDE).all().all()
assert (sq2_groups.loc[sq2_groups["eligible"], ["n_treated", "n_control"]] >= MIN_GAMES_PER_SIDE).all().all()

In [ ]:
# 3.4d · Check the numbers reported in the notes and the README
# Reason: the text quotes these numbers -> a rerun that gives others must stop

# the run reproduces the numbers given in the Check notes and in the README
assert (len(raw_a), len(raw_c)) == (125_855, 85_103)
assert (len(games_clean), int(games_clean["has_zh"].sum())) == (45_445, 14_701)
assert len(cn_excluded) == 2_030
assert (len(sq1_groups), int(sq1_groups["eligible"].sum())) == (360, 190)
assert sq2_panel["arm"].value_counts().to_dict() == {"control": 20_839, "treated": 510, "excluded": 241}
assert (len(sq2_groups), int(sq2_groups["eligible"].sum())) == (24, 16)
assert audit["signal"].value_counts().reindex(SIGNALS).tolist() == [25, 12, 4, 9]
print("all checks passed")

## 4.​ Data analysis / EDA

- Describes the three saved tables only; no comparison of the two sides (that is section 5).
- Reads only the saved tables; figures go to `report/figures/`.
- To run it alone: cell I.a, then 4a.

| Section | Reads |
| :--- | :--- |
| 4.1 Games | `pcd_games_clean` |
| 4.2 SQ1 groups | `pcd_sq1_groups` |
| 4.3 SQ2 groups | `pcd_sq2_groups` |

In [ ]:
# 4a · Load the saved tables
# Reason: Stage 4 reads only the saved tables -> can run on its own

import matplotlib.pyplot as plt

games_clean = pd.read_csv(PROCESSED_CSV + PREFIX + "games_clean.csv")
sq1_groups = pd.read_csv(PROCESSED_CSV + PREFIX + "sq1_groups.csv")
sq2_groups = pd.read_csv(PROCESSED_CSV + PREFIX + "sq2_groups.csv")
print(f"games_clean {len(games_clean):,} rows | sq1_groups {len(sq1_groups)} | sq2_groups {len(sq2_groups)}")

### 4.1 Games: `pcd_games_clean`

### 4.2 SQ1 groups: `pcd_sq1_groups`

### 4.3 SQ2 groups: `pcd_sq2_groups`

## 5.​ Data modelling

- Compares the two sides of each group and answers SQ1, SQ2, SQ3.
- Reads the saved tables loaded in 4a; figures go to `report/figures/`.
- Exception: 5.1b also needs section 3 in memory (Run All).

| Section | Reads |
| :--- | :--- |
| 5.1 SQ1 | `pcd_sq1_groups` |
| 5.2 SQ2 | `pcd_sq2_groups` |
| 5.3 SQ3 | 5.1 and 5.2, per genre |

### 5.1 SQ1: how much larger is the reach of games with Simplified Chinese, and in which genres?

#### Does the SQ1 gap depend on the Chinese-first filter? (5.1b)

- **Reason:** the filter misses some Chinese games (Black Myth: Wukong).
- **Rule:** build `pcd_games_clean` and the SQ1 groups again with no filter, a stricter filter, and without the top 1% games; compute the gap each time.
- Gap of a group = median with Chinese / median without - 1; overall gap = median over eligible groups.
- Runs on copies → nothing is changed or saved.

In [ ]:
# 5.1b · Compute the SQ1 gap again under other filters
# Reason: the filter misses some Chinese games -> test whether that changes the result
# Rule: no filter, a stricter filter, and without the top 1% games
# Needs: feat_a and the functions of section 3 (Run All)

TOP_SHARE = 0.01              # share of most-reviewed games dropped in the last row


def sq1_gap(groups):
    """Median over eligible groups of (median with Chinese / median without - 1)."""
    eligible = groups[groups["eligible"]]
    pct_diff = eligible["median_reviews_zh"] / eligible["median_reviews_no_zh"] - 1
    return round(float(pct_diff.median()), 3)


# Works on a copy -> feat_a and the saved tables are not changed.
def sq1_summary(label, is_cn_game, drop_top=0.0):
    """SQ1 under the given Chinese-first flag, optionally without the most-reviewed games."""
    games, _ = build_games_clean(feat_a.assign(is_cn_game=is_cn_game))
    if drop_top:
        games = games[games["total_reviews"] <= games["total_reviews"].quantile(1 - drop_top)]
    groups = build_sq1_groups(add_sq1_group(games))
    return {"filter": label, "games": len(games), "eligible_groups": int(groups["eligible"].sum()),
            "median_gap": sq1_gap(groups)}


chosen = feat_a["cn_text"] | feat_a["cn_dev_name"]
stricter = chosen | feat_a["cn_dev_kana"] | feat_a["cn_audio"]
sensitivity = pd.DataFrame([
    sq1_summary("no filter", chosen & False),
    sq1_summary("chosen: text or developer name", chosen),
    sq1_summary("stricter: + kana names and voice-over", stricter),
    sq1_summary("chosen, without the 1% most-reviewed games", chosen, drop_top=TOP_SHARE),
])

# check: the "chosen" row = the saved tables; the gaps quoted in the README
assert sensitivity.loc[1, "games"] == len(games_clean)
assert sensitivity.loc[1, "median_gap"] == sq1_gap(sq1_groups)
assert sensitivity["median_gap"].tolist() == [1.254, 1.2, 1.174, 1.121]
sensitivity

**Result**
- Median gap: +125% (no filter), +120% (chosen), +117% (stricter), +112% (without top 1%).
- Same size in every row → the answer to SQ1 does not depend on the filter.

### 5.2 SQ2: does adding Simplified Chinese after launch still increase reach, and in which genres?

### 5.3 SQ3: for each genre, translate at launch, later, or not a priority?

## 6.​ Model deployment